# 🔍 People Analytics: Predicción de Rotación Voluntaria

**Dataset:** IBM HR Analytics Employee Attrition & Performance (1,470 empleados · 35 variables)  
**Objetivo:** Identificar patrones sociológicos y entrenar modelos predictivos para anticipar rotación voluntaria.  
**Stack:** Python · Pandas · Scikit-learn · XGBoost · Matplotlib

---
## Contenido
1. Carga y limpieza · 2. EDA · 3. Feature Engineering · 4. Modelado · 5. Evaluación · 6. Impacto económico · 7. Conclusiones

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, matplotlib.patches as mpatches
import warnings; warnings.filterwarnings('ignore')
from scipy.stats import gaussian_kde
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (roc_auc_score, roc_curve, f1_score, confusion_matrix,
                              average_precision_score, precision_recall_curve, classification_report)
from xgboost import XGBClassifier
print('✅ Librerías listas')

## 1. Carga y Limpieza de Datos

In [ ]:
df = pd.read_csv('data/WA_Fn-UseC_-HR-Employee-Attrition.csv')
print(f'Shape: {df.shape} | Attrition: {df["Attrition"].value_counts().to_dict()}')
df.drop(columns=['EmployeeCount','Over18','StandardHours','EmployeeNumber'], inplace=True)
df['Attrition_bin'] = (df['Attrition']=='Yes').astype(int)
df['OverTime_bin']  = (df['OverTime']=='Yes').astype(int)
for col in ['BusinessTravel','Department','EducationField','Gender','JobRole','MaritalStatus']:
    df[col+'_enc'] = LabelEncoder().fit_transform(df[col])
print(f'Tasa de rotación: {df["Attrition_bin"].mean():.1%}')

## 2. Análisis Exploratorio

**Hallazgos clave:**
- Tasa global: **16.1%** (clases desbalanceadas)
- Sales tiene la mayor tasa por departamento (20.6%)
- Sales Representatives lideran por rol (39.8%)
- Horas extra es el factor individual más discriminante

In [ ]:
from IPython.display import Image, display
display(Image('figures/01_attrition_overview.png'))
display(Image('figures/04_eda_key_vars.png'))

## 3. Feature Engineering — Perspectiva Sociológica

| Variable | Marco teórico |
|---|---|
| `SatisfactionIndex` | Herzberg — satisfacción multidimensional |
| `OverloadScore` | Karasek — modelo demanda-control |
| `StagnationScore` | Sicherman — movilidad interna |
| `IncomeGap` | Adams — teoría de equidad |

In [ ]:
df['SatisfactionIndex'] = (df['JobSatisfaction']+df['EnvironmentSatisfaction']+
                            df['RelationshipSatisfaction']+df['WorkLifeBalance'])/16
df['OverloadScore']  = ((df['OverTime_bin']*2)+(df['NumCompaniesWorked']>3).astype(int)+
                        (df['WorkLifeBalance']<=2).astype(int))/4
df['StagnationScore']= ((df['YearsSinceLastPromotion']>3).astype(int)+
                        (df['YearsInCurrentRole']>df['YearsAtCompany']*0.6).astype(int)+
                        (df['JobLevel']==1).astype(int))/3
df['IncomeGap'] = df.groupby(['JobRole','JobLevel'])['MonthlyIncome'].transform(
    lambda x:(x-x.median())/(x.std()+1)).fillna(0)

def classify_profile(row):
    if row['OverTime_bin']==1 and row['WorkLifeBalance']<=2 and row['JobSatisfaction']<=2: return 'Quemado'
    elif row['SatisfactionIndex']<0.45 and row['JobLevel']<=2: return 'Desenganchado'
    elif row['YearsSinceLastPromotion']>=4 and row['YearsAtCompany']>=5 and row['PerformanceRating']>=3: return 'Estancado'
    else: return 'Estable'
df['Perfil'] = df.apply(classify_profile, axis=1)
print(df.groupby('Perfil')['Attrition_bin'].agg(['count','mean']).rename(columns={'count':'N','mean':'Tasa'}).round(3))
display(Image('figures/03_perfiles_sociologicos.png'))

## 4. Modelado Predictivo

In [ ]:
feature_cols = ['Age','DistanceFromHome','Education','EnvironmentSatisfaction','JobInvolvement',
    'JobLevel','JobSatisfaction','MonthlyIncome','NumCompaniesWorked','OverTime_bin',
    'PercentSalaryHike','RelationshipSatisfaction','StockOptionLevel','TotalWorkingYears',
    'TrainingTimesLastYear','WorkLifeBalance','YearsAtCompany','YearsInCurrentRole',
    'YearsSinceLastPromotion','YearsWithCurrManager','SatisfactionIndex','OverloadScore',
    'StagnationScore','IncomeGap','BusinessTravel_enc','Department_enc','EducationField_enc',
    'Gender_enc','JobRole_enc','MaritalStatus_enc']
X=df[feature_cols].fillna(0); y=df['Attrition_bin']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
scaler=StandardScaler(); X_train_sc=scaler.fit_transform(X_train); X_test_sc=scaler.transform(X_test)
scale_pos=(y_train==0).sum()/(y_train==1).sum()
models_cfg={'Logistic Regression':LogisticRegression(max_iter=1000,class_weight='balanced',C=0.5,random_state=42),
            'Random Forest':RandomForestClassifier(n_estimators=300,class_weight='balanced',max_depth=8,min_samples_leaf=5,random_state=42),
            'XGBoost':XGBClassifier(n_estimators=300,max_depth=5,learning_rate=0.05,scale_pos_weight=scale_pos,eval_metric='logloss',random_state=42,verbosity=0)}
results={}
for name,model in models_cfg.items():
    X_fit=X_train_sc if name=='Logistic Regression' else X_train.values
    X_eval=X_test_sc if name=='Logistic Regression' else X_test.values
    model.fit(X_fit,y_train); proba=model.predict_proba(X_eval)[:,1]; pred=(proba>=0.35).astype(int)
    results[name]={'model':model,'proba':proba,'pred':pred,'auc':roc_auc_score(y_test,proba),'f1':f1_score(y_test,pred),'ap':average_precision_score(y_test,proba)}
    print(f"{name:25s} | AUC={results[name]['auc']:.4f} | F1={results[name]['f1']:.4f} | AP={results[name]['ap']:.4f}")

## 5. Evaluación

In [ ]:
from IPython.display import Image, display
display(Image('figures/05_roc_curves.png'))
display(Image('figures/07_confusion_matrix.png'))
best_name=max(results,key=lambda k:results[k]['auc'])
print(f'\n=== {best_name} ===')
print(classification_report(y_test,results[best_name]['pred'],target_names=['No rotó','Rotó']))

## 6. Importancia de Variables

In [ ]:
from IPython.display import Image, display
display(Image('figures/06_feature_importance.png'))

## 7. Impacto Económico

**Supuestos:** Costo de reemplazo = $15,000 USD | Programa retención = $500/persona | Tasa retención exitosa = 60%

| Perfil | Intervención |
|---|---|
| 🔥 Quemado | Reducir horas extra, revisar carga |
| 😶 Desenganchado | Re-engagement, 1:1 frecuente |
| ⏳ Estancado | Plan de carrera explícito |
| ✅ Estable | Reconocimiento y cultura |

In [ ]:
from IPython.display import Image, display
display(Image('figures/08_risk_business_impact.png'))
X_all_sc=scaler.transform(df[feature_cols].fillna(0))
df['RiskScore']=results['Logistic Regression']['model'].predict_proba(X_all_sc)[:,1]
high_risk=df[df['RiskScore']>=0.35]
print(f'Alto riesgo: {len(high_risk)} empleados ({len(high_risk)/len(df):.1%})')
print(f'De ellos, realmente rotan: {high_risk["Attrition_bin"].mean():.1%}')
print(df[df['RiskScore']>=0.35].groupby('Department')['RiskScore'].agg(['count','mean']).round(3).sort_values('mean',ascending=False))

## 8. Conclusiones

1. **Horas extra** = predictor más poderoso (3.4× más riesgo)
2. **Ingreso mensual** actúa como factor protector (+$5,500 de diferencia media)
3. **Los primeros 3 años** son la ventana de intervención más crítica
4. **Perfiles sociológicos** capturan dinámicas que las variables crudas no miden
5. **Logistic Regression** (AUC 0.82) es el modelo más interpretable para equipos de RRHH

### Próximos pasos
- [ ] Survival Analysis con Cox Proportional Hazards
- [ ] SHAP values para explicabilidad individual
- [ ] Dashboard en Streamlit